# Estatística com R

Testes de hipóteses, regressão e intervalos de confiança utilizando conjuntos de dados integrados do R.

Nenhum download de dados ou instalação de pacotes é necessária — utiliza apenas o R básico (Base R).

## 1. Estatística Descritiva

In [ ]:
data(mtcars)
cat("Conjunto de dados: mtcars (", nrow(mtcars), " carros, ", ncol(mtcars), " variáveis)\n\n")
summary(mtcars[, c("mpg", "hp", "wt", "disp")])

## 2. Teste t para Duas Amostras

Carros com transmissão manual têm melhor rendimento de combustível (MPG) do que os automáticos?

In [ ]:
auto <- mtcars$mpg[mtcars$am == 0]
manual <- mtcars$mpg[mtcars$am == 1]

cat("Automático:", round(mean(auto), 1), "MPG (N =", length(auto), ")\n")
cat("Câmbio manual:   ", round(mean(manual), 1), "MPG (N =", length(manual), ")\n\n")

t_result <- t.test(manual, auto, alternative = "greater")
print(t_result)

cat("\nConclusão:",
    ifelse(t_result$p.value < 0.05,
           "Rejeitar H0 — carros manuais têm MPG significativamente maior",
           "Falha ao rejeitar H0"))

## 3. Teste Qui-Quadrado

A quantidade de cilindros e o tipo de transmissão são independentes?

In [ ]:
tab <- table(Cylinders = mtcars$cyl, Transmission = mtcars$am)
colnames(tab) <- c("Automático", "Manual")
print(tab)
cat("\n")
chisq.test(tab)

## 4. Regressão Linear Múltipla

In [ ]:
model <- lm(mpg ~ wt + hp + am, data = mtcars)
summary(model)

## 5. Diagnóstico de Regressão

In [ ]:
par(mfrow = c(2, 2))
plot(model)

## 6. Intervalos de Confiança

In [ ]:
ci <- confint(model, level = 0.95)
cat("Intervalos de confiança de 95%:\n")
print(round(ci, 4))

In [ ]:
coefs <- coef(model)[-1]
ci_vals <- ci[-1, ]
n <- length(coefs)

par(mfrow = c(1, 1), mar = c(5, 8, 4, 2))
plot(coefs, 1:n, xlim = range(ci_vals),
     pch = 19, col = "#58a6ff", cex = 1.5,
     yaxt = "n", xlab = "Estimativa", ylab = "",
     main = "Intervalos de confiança de 95% para os coeficientes")
axis(2, at = 1:n, labels = names(coefs), las = 1)
segments(ci_vals[, 1], 1:n, ci_vals[, 2], 1:n,
         lwd = 3, col = "#58a6ff")
abline(v = 0, lty = 2, col = "#f85149", lwd = 1.5)

## 7. ANOVA Unidirecional (One-Way ANOVA)

O rendimento (MPG) difere significativamente entre as contagens de cilindros?

In [ ]:
anova_model <- aov(mpg ~ factor(cyl), data = mtcars)
summary(anova_model)
cat("\nComparações post-hoc de Tukey HSD:\n")
TukeyHSD(anova_model)

In [ ]:
boxplot(mpg ~ cyl, data = mtcars,
        main = "MPG por contagem de cilindros",
        xlab = "Cilindros", ylab = "Milhas por galão (MPG)",
        col = c("#58a6ff", "#a371f7", "#f85149"))

## Resumo

- **Teste t de Welch**: Na comparação unilateral não ajustada, carros com transmissão manual apresentam maior média de MPG
- **Qui-quadrado**: A tabela de contingência sugere uma associação, mas as baixas contagens esperadas disparam um aviso de aproximação, portanto interprete este resultado com cautela
- **Regressão**: Peso e potência (horsepower) são preditores negativos significativos após o ajuste; o tipo de transmissão não é significativo neste modelo
- **ANOVA**: O MPG difere significativamente entre os grupos de 4, 6 e 8 cilindros; os resultados do teste de Tukey identificam as diferenças pareadas